# Absolute Cost Estimation — Evaluation

Evaluate the **Embedding + KNN** approach for estimating absolute costs (USD) across models.

- **Method**: For a given query, embed it, find K nearest neighbors in the training set, and use their actual costs as the estimate.
- **Baseline**: Query-independent mean cost per model from the training set.
- **Evaluation**: Standard train-test split, measuring MAE and MAPE.

In [1]:
import sys, os, importlib
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'ThinkingTax'))
# Also handle running from the repo root
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import method.absolute_cost_estimator as _ace_mod
importlib.reload(_ace_mod)
from method.absolute_cost_estimator import AbsoluteCostEstimator

## 1. Configuration

In [2]:
DATA_DIR = "../data/consolidated"
EMBEDDING_PROVIDER = "gemini"
K = 5
TEST_RATIO = 0.2
SEED = 42

# Load API key
with open("../apikey.json") as f:
    api_keys = json.load(f)

API_KEY = api_keys["gemini_api_key"]

# Load models from experiment config (only use these 8 models)
with open("../constant/experiment_config.json") as f:
    exp_config = json.load(f)

MODELS = [m["model_name"] for m in exp_config["models"]]
DATASETS = [d["file_prefix"] for d in exp_config["datasets"]]

# Load model pricing info
with open("../constant/model_info.json") as f:
    model_info = json.load(f)

MODEL_INFO = model_info["models"]
short_names = {m["model_name"]: m["short_name"] for m in MODEL_INFO}

print(f"Using {len(MODELS)} models: {MODELS}")
print(f"Using {len(DATASETS)} datasets: {DATASETS}")

Using 8 models: ['gpt-5.2-high', 'gpt-5-mini', 'gemini-3.1-pro-preview', 'gemini-3-flash-preview', 'claude-opus-4.6-thinking', 'claude-haiku-4.5', 'kimi-k2.5', 'MiniMax-M2.5']
Using 9 datasets: ['aime-hybrid', 'arc-agi-v1', 'arenahard-test', 'gpqa-test', 'hle-test', 'livecodebench-test', 'livemathbench-test', 'mmlupro-test_3000', 'simpleqa-test']


## 2. Build Index

In [3]:
estimator = AbsoluteCostEstimator(
    models=MODELS,
    embedding_provider=EMBEDDING_PROVIDER,
    api_key=API_KEY,
    k=K,
)
estimator.build_index(data_dir=DATA_DIR, datasets=DATASETS)
print(estimator)
print(f"Models: {estimator.model_list}")
print(f"Total indexed queries: {estimator.n_queries}")

AbsoluteCostEstimator(models=8, provider='gemini', k=5, index=built, n_queries=11476)
Models: ['gpt-5.2-high', 'gpt-5-mini', 'gemini-3.1-pro-preview', 'gemini-3-flash-preview', 'claude-opus-4.6-thinking', 'claude-haiku-4.5', 'kimi-k2.5', 'MiniMax-M2.5']
Total indexed queries: 11476


## 3. Train-Test Evaluation

In [4]:
# KNN (direct cost)
mae_knn = estimator.evaluate(test_ratio=TEST_RATIO, seed=SEED, metric="mae")

# KNN (predict tokens → pricing formula)
mae_token = estimator.evaluate_token_based(MODEL_INFO, test_ratio=TEST_RATIO, seed=SEED)

# Mean baseline
mae_baseline = estimator.mean_baseline(test_ratio=TEST_RATIO, seed=SEED)

eval_df = pd.DataFrame({
    "Model": [short_names.get(m, m) for m in estimator.model_list],
    "Mean Baseline ($)": [f"{mae_baseline[m]:.6f}" for m in estimator.model_list],
    "KNN Cost ($)": [f"{mae_knn[m]:.6f}" for m in estimator.model_list],
    "KNN Token ($)": [f"{mae_token[m]:.6f}" for m in estimator.model_list],
})
print("=== Per-Model MAE Comparison (Absolute Cost, Train-Test Split) ===")
print(f"Test ratio: {TEST_RATIO}, K: {K}, Seed: {SEED}\n")
print(eval_df.to_string(index=False))
print()
avg_baseline = np.mean(list(mae_baseline.values()))
avg_knn = np.mean(list(mae_knn.values()))
avg_token = np.mean(list(mae_token.values()))
print(f"Average MAE — Mean Baseline: ${avg_baseline:.6f},  KNN Cost: ${avg_knn:.6f},  KNN Token: ${avg_token:.6f}")

=== Per-Model MAE Comparison (Absolute Cost, Train-Test Split) ===
Test ratio: 0.2, K: 5, Seed: 42

           Model Mean Baseline ($) KNN Cost ($) KNN Token ($)
         GPT-5.2          0.048995     0.045783      0.045783
      GPT-5 Mini          0.003911     0.002713      0.002713
  Gemini 3.1 Pro          0.111688     0.094119      0.094119
  Gemini 3 Flash          0.055506     0.034409      0.034409
 Claude Opus 4.6          0.072128     0.051972      0.051972
Claude Haiku 4.5          0.002299     0.001251      0.001251
       Kimi K2.5          0.023310     0.013797      0.013797
    MiniMax-M2.5          0.000828     0.000591      0.000591

Average MAE — Mean Baseline: $0.039833,  KNN Cost: $0.030579,  KNN Token: $0.030579


## 3b. Method Comparison: Exploring Better Approaches

In [5]:
"""
Compare multiple approaches for absolute cost estimation.
All use the same train/test split for fair comparison.
"""
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA

train_idx, test_idx = estimator.train_test_split(TEST_RATIO, SEED)
train_embs = estimator._embeddings[train_idx]
train_costs = estimator._costs[train_idx]
test_embs = estimator._embeddings[test_idx]
test_costs = estimator._costs[test_idx]
train_pt = estimator._prompt_tokens[train_idx]
test_pt = estimator._prompt_tokens[test_idx]
n_models = len(estimator.model_list)

# --- Helper: compute per-model MAE ---
def compute_mae(predicted, actual):
    """predicted, actual: (n_test, n_models)"""
    return np.abs(predicted - actual).mean(axis=0)

results = {}

# =============================================
# 1. Mean Baseline (query-independent)
# =============================================
mean_pred = np.tile(train_costs.mean(axis=0), (len(test_idx), 1))
results["Mean Baseline"] = compute_mae(mean_pred, test_costs)

# =============================================
# 2. KNN (current method, K=5)
# =============================================
sim_matrix = estimator._cosine_similarity(test_embs, train_embs)
k = min(K, len(train_idx))
knn_pred = np.zeros_like(test_costs)
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k = np.argsort(sims)[-k:][::-1]
    w = sims[top_k]; w = w / w.sum()
    knn_pred[i] = np.average(train_costs[top_k], axis=0, weights=w)
results["KNN (K=5)"] = compute_mae(knn_pred, test_costs)

# =============================================
# 3. KNN in log-space
# =============================================
train_log = np.log1p(train_costs)  # log(1+cost) to handle near-zero
knn_log_pred = np.zeros_like(test_costs)
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k = np.argsort(sims)[-k:][::-1]
    w = sims[top_k]; w = w / w.sum()
    log_est = np.average(train_log[top_k], axis=0, weights=w)
    knn_log_pred[i] = np.expm1(log_est)  # back to cost space
results["KNN log-space"] = compute_mae(knn_log_pred, test_costs)

# =============================================
# 4. KNN with median (more robust to outliers)
# =============================================
knn_med_pred = np.zeros_like(test_costs)
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k = np.argsort(sims)[-k:][::-1]
    knn_med_pred[i] = np.median(train_costs[top_k], axis=0)
results["KNN median"] = compute_mae(knn_med_pred, test_costs)

# =============================================
# 5. Ridge Regression on embeddings
# =============================================
ridge_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = Ridge(alpha=1.0).fit(train_embs, train_costs[:, j])
    ridge_pred[:, j] = model.predict(test_embs)
ridge_pred = np.maximum(ridge_pred, 0)  # cost >= 0
results["Ridge Regression"] = compute_mae(ridge_pred, test_costs)

# =============================================
# 6. Ridge + query length feature
# =============================================
# Use mean prompt tokens across models as "query length" proxy
train_ql = train_pt.mean(axis=1, keepdims=True)  # (n_train, 1)
test_ql = test_pt.mean(axis=1, keepdims=True)
train_aug = np.hstack([train_embs, train_ql / train_ql.max()])
test_aug = np.hstack([test_embs, test_ql / train_ql.max()])

ridge_ql_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = Ridge(alpha=1.0).fit(train_aug, train_costs[:, j])
    ridge_ql_pred[:, j] = model.predict(test_aug)
ridge_ql_pred = np.maximum(ridge_ql_pred, 0)
results["Ridge + query len"] = compute_mae(ridge_ql_pred, test_costs)

# =============================================
# 7. Ridge in log-space
# =============================================
ridge_log_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = Ridge(alpha=1.0).fit(train_embs, np.log1p(train_costs[:, j]))
    ridge_log_pred[:, j] = np.expm1(model.predict(test_embs))
ridge_log_pred = np.maximum(ridge_log_pred, 0)
results["Ridge log-space"] = compute_mae(ridge_log_pred, test_costs)

# =============================================
# 8. PCA + Ridge (reduce dim to 50)
# =============================================
pca = PCA(n_components=50, random_state=SEED).fit(train_embs)
train_pca = pca.transform(train_embs)
test_pca = pca.transform(test_embs)
pca_ridge_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = Ridge(alpha=1.0).fit(train_pca, train_costs[:, j])
    pca_ridge_pred[:, j] = model.predict(test_pca)
pca_ridge_pred = np.maximum(pca_ridge_pred, 0)
results["PCA50 + Ridge"] = compute_mae(pca_ridge_pred, test_costs)

# =============================================
# 9. Random Forest on PCA embeddings
# =============================================
rf_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)
    model.fit(train_pca, train_costs[:, j])
    rf_pred[:, j] = model.predict(test_pca)
results["PCA50 + RF"] = compute_mae(rf_pred, test_costs)

# =============================================
# 10. Gradient Boosting on PCA embeddings
# =============================================
gb_pred = np.zeros_like(test_costs)
for j in range(n_models):
    model = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=SEED)
    model.fit(train_pca, train_costs[:, j])
    gb_pred[:, j] = model.predict(test_pca)
results["PCA50 + GBR"] = compute_mae(gb_pred, test_costs)

# =============================================
# 11. KNN + Ridge ensemble (average predictions)
# =============================================
ensemble_pred = (knn_pred + ridge_pred) / 2
results["KNN+Ridge ensemble"] = compute_mae(ensemble_pred, test_costs)

# =============================================
# Print results
# =============================================
print("=== Method Comparison: Per-Model MAE ($) ===\n")
model_names = [short_names.get(m, m) for m in estimator.model_list]

# Build table
rows = []
for method_name, maes in results.items():
    row = {"Method": method_name}
    for i, m in enumerate(estimator.model_list):
        row[short_names.get(m, m)] = maes[i]
    row["Average"] = maes.mean()
    rows.append(row)

cmp_df = pd.DataFrame(rows)
# Sort by average MAE
cmp_df = cmp_df.sort_values("Average").reset_index(drop=True)

# Format for display
display_cols = ["Method"] + model_names + ["Average"]
print(cmp_df[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()

# Highlight best method
best = cmp_df.iloc[0]
print(f"Best method: {best['Method']} (avg MAE = ${best['Average']:.6f})")

=== Method Comparison: Per-Model MAE ($) ===

            Method  GPT-5.2  GPT-5 Mini  Gemini 3.1 Pro  Gemini 3 Flash  Claude Opus 4.6  Claude Haiku 4.5  Kimi K2.5  MiniMax-M2.5  Average
        KNN median   0.0366      0.0025          0.0728          0.0308           0.0455            0.0010     0.0130        0.0006   0.0253
       PCA50 + GBR   0.0402      0.0024          0.0889          0.0328           0.0475            0.0011     0.0118        0.0006   0.0281
        PCA50 + RF   0.0401      0.0024          0.0888          0.0328           0.0470            0.0011     0.0123        0.0006   0.0281
   Ridge log-space   0.0405      0.0026          0.0865          0.0346           0.0476            0.0014     0.0124        0.0006   0.0283
KNN+Ridge ensemble   0.0424      0.0025          0.0906          0.0340           0.0485            0.0012     0.0125        0.0006   0.0290
 Ridge + query len   0.0418      0.0026          0.0919          0.0350           0.0486            0.0013  

In [6]:
print(f"knn_med_pred shape: {knn_med_pred.shape}")
print(f"test_costs shape: {test_costs.shape}")

knn_med_pred shape: (2295, 8)
test_costs shape: (2295, 8)


In [7]:
"""
Round 2: More targeted experiments building on KNN median as the best so far.
"""
"""
from scipy.stats import trim_mean as _trim_mean

results2 = {}

# Reuse train/test from previous cell
# train_embs, test_embs, train_costs, test_costs, sim_matrix already available

# =============================================
# Baselines for reference
# =============================================
results2["Mean Baseline"] = compute_mae(np.tile(train_costs.mean(axis=0), (len(test_idx), 1)), test_costs)
results2["KNN wmean (K=5)"] = compute_mae(knn_pred, test_costs)
results2["KNN median (K=5)"] = compute_mae(knn_med_pred, test_costs)

# =============================================
# 1. KNN median with different K values
# =============================================
for k_val in [3, 7, 10, 15, 20]:
    pred = np.zeros_like(test_costs)
    for i in range(len(test_idx)):
        sims = sim_matrix[i]
        kk = min(k_val, len(train_idx))
        top_k = np.argsort(sims)[-kk:][::-1]
        pred[i] = np.median(train_costs[top_k], axis=0)
    results2[f"KNN median (K={k_val})"] = compute_mae(pred, test_costs)

# =============================================
# 2. KNN trimmed mean (drop top/bottom 20%)
# =============================================
for k_val in [5, 10, 15, 20]:
    pred = np.zeros_like(test_costs)
    for i in range(len(test_idx)):
        sims = sim_matrix[i]
        kk = min(k_val, len(train_idx))
        top_k = np.argsort(sims)[-kk:][::-1]
        for j in range(n_models):
            pred[i, j] = _trim_mean(train_costs[top_k, j], proportiontocut=0.2)
    results2[f"KNN trim20 (K={k_val})"] = compute_mae(pred, test_costs)

# =============================================
# 3. Weighted median (similarity-weighted quantile)
# =============================================
def weighted_median(values, weights):
    """Compute weighted median."""
    order = np.argsort(values)
    sorted_vals = values[order]
    sorted_w = weights[order]
    cum_w = np.cumsum(sorted_w)
    cum_w /= cum_w[-1]
    idx = np.searchsorted(cum_w, 0.5)
    return sorted_vals[min(idx, len(sorted_vals) - 1)]

for k_val in [5, 10, 15, 20]:
    pred = np.zeros_like(test_costs)
    for i in range(len(test_idx)):
        sims = sim_matrix[i]
        kk = min(k_val, len(train_idx))
        top_k = np.argsort(sims)[-kk:][::-1]
        w = sims[top_k]
        w = np.maximum(w, 0)
        if w.sum() == 0:
            w = np.ones_like(w)
        for j in range(n_models):
            pred[i, j] = weighted_median(train_costs[top_k, j], w)
    results2[f"KNN wmedian (K={k_val})"] = compute_mae(pred, test_costs)

# =============================================
# 4. KNN median in log-space
# =============================================
for k_val in [5, 10]:
    pred = np.zeros_like(test_costs)
    train_log = np.log1p(train_costs)
    for i in range(len(test_idx)):
        sims = sim_matrix[i]
        kk = min(k_val, len(train_idx))
        top_k = np.argsort(sims)[-kk:][::-1]
        pred[i] = np.expm1(np.median(train_log[top_k], axis=0))
    results2[f"KNN log-median (K={k_val})"] = compute_mae(pred, test_costs)

# =============================================
# 5. Adaptive K: use all neighbors with sim > threshold
# =============================================
for thresh in [0.7, 0.8, 0.85, 0.9]:
    pred = np.zeros_like(test_costs)
    for i in range(len(test_idx)):
        sims = sim_matrix[i]
        mask = sims >= thresh
        if mask.sum() < 3:
            top_k = np.argsort(sims)[-3:][::-1]
        else:
            top_k = np.where(mask)[0]
        pred[i] = np.median(train_costs[top_k], axis=0)
    results2[f"Adaptive median (t={thresh})"] = compute_mae(pred, test_costs)

# =============================================
# 6. KNN median + Ridge ensemble
# =============================================
pred_med5 = np.zeros_like(test_costs)
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k = np.argsort(sims)[-5:][::-1]
    pred_med5[i] = np.median(train_costs[top_k], axis=0)
ensemble2 = (pred_med5 + ridge_pred) / 2
results2["KNN-med+Ridge ens"] = compute_mae(ensemble2, test_costs)

# =============================================
# 7. Query length (char count) regression baseline
# =============================================
train_qlen = np.array([len(estimator._queries[i]) for i in train_idx]).reshape(-1, 1)
test_qlen = np.array([len(estimator._queries[i]) for i in test_idx]).reshape(-1, 1)
qlen_pred = np.zeros_like(test_costs)
for j in range(n_models):
    m = Ridge(alpha=1.0).fit(train_qlen, train_costs[:, j])
    qlen_pred[:, j] = np.maximum(m.predict(test_qlen), 0)
results2["Query length Ridge"] = compute_mae(qlen_pred, test_costs)

# =============================================
# 8. GBR on full embeddings (not PCA)
# =============================================
gb_full_pred = np.zeros_like(test_costs)
for j in range(n_models):
    m = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                   subsample=0.8, random_state=SEED)
    m.fit(train_embs, train_costs[:, j])
    gb_full_pred[:, j] = m.predict(test_embs)
results2["GBR full embed"] = compute_mae(gb_full_pred, test_costs)

# =============================================
# Print results
# =============================================
print("=== Round 2: Method Comparison ($) ===\n")
model_names = [short_names.get(m, m) for m in estimator.model_list]

rows2 = []
for method_name, maes in results2.items():
    row = {"Method": method_name, "Average": maes.mean()}
    rows2.append(row)

cmp2 = pd.DataFrame(rows2).sort_values("Average").reset_index(drop=True)
print(cmp2.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print()
print(f"Best: {cmp2.iloc[0]['Method']} (avg MAE = ${cmp2.iloc[0]['Average']:.6f})")
"""

SyntaxError: invalid syntax (2981065849.py, line 48)

## 4. Sensitivity Analysis: Varying K

In [ ]:
k_values = [1, 3, 5, 10, 20, 50]
k_mae = {}

for k_val in k_values:
    estimator.k = k_val
    result = estimator.evaluate(test_ratio=TEST_RATIO, seed=SEED, metric="mae")
    k_mae[k_val] = np.mean(list(result.values()))

# Restore original K
estimator.k = K

k_df = pd.DataFrame({"K": list(k_mae.keys()), "Avg MAE ($)": [f"{v:.6f}" for v in k_mae.values()]})
print("=== K Sensitivity (Absolute Cost) ===")
print(k_df.to_string(index=False))

=== K Sensitivity (Absolute Cost) ===
 K Avg MAE ($)
 1    0.033160
 3    0.030297
 5    0.030579
10    0.030831
20    0.031755
50    0.033949


## 5. Sensitivity Analysis: Varying Test Ratio

In [ ]:
test_ratios = [0.1, 0.2, 0.3, 0.5]
ratio_mae = {}

for tr in test_ratios:
    result = estimator.evaluate(test_ratio=tr, seed=SEED, metric="mae")
    ratio_mae[tr] = np.mean(list(result.values()))

ratio_df = pd.DataFrame({
    "Test Ratio": list(ratio_mae.keys()),
    "Avg MAE ($)": [f"{v:.6f}" for v in ratio_mae.values()],
})
print("=== Test Ratio Sensitivity (Absolute Cost) ===")
print(ratio_df.to_string(index=False))

=== Test Ratio Sensitivity (Absolute Cost) ===
 Test Ratio Avg MAE ($)
        0.1    0.031122
        0.2    0.030579
        0.3    0.030907
        0.5    0.032230


## 6. Per-Query Error Distribution

In [ ]:
# Per-query MAE for both methods
train_idx, test_idx = estimator.train_test_split(TEST_RATIO, SEED)

train_embs = estimator._embeddings[train_idx]
train_costs = estimator._costs[train_idx]
test_embs = estimator._embeddings[test_idx]
test_costs = estimator._costs[test_idx]

# --- KNN per-query MAE ---
sim_matrix = estimator._cosine_similarity(test_embs, train_embs)
k = min(estimator.k, len(train_idx))

per_query_mae_knn = []
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k_idx = np.argsort(sims)[-k:][::-1]
    top_k_sims = sims[top_k_idx]
    weights = top_k_sims / top_k_sims.sum()
    predicted = np.average(train_costs[top_k_idx], axis=0, weights=weights)
    actual = test_costs[i]
    mae = np.mean(np.abs(predicted - actual))
    per_query_mae_knn.append(mae)

per_query_mae_knn = np.array(per_query_mae_knn)

# --- Mean baseline per-query MAE ---
mean_costs = train_costs.mean(axis=0)  # (n_models,)
per_query_mae_baseline = np.mean(np.abs(test_costs - mean_costs[None, :]), axis=1)

print(f"Per-query MAE statistics (n={len(test_idx)}):")
print(f"{'':>10s}  {'Baseline':>12s}  {'KNN':>12s}")
print(f"{'Mean':>10s}  ${per_query_mae_baseline.mean():11.6f}  ${per_query_mae_knn.mean():11.6f}")
print(f"{'Median':>10s}  ${np.median(per_query_mae_baseline):11.6f}  ${np.median(per_query_mae_knn):11.6f}")
print(f"{'Std':>10s}  ${per_query_mae_baseline.std():11.6f}  ${per_query_mae_knn.std():11.6f}")
print(f"{'P90':>10s}  ${np.percentile(per_query_mae_baseline, 90):11.6f}  ${np.percentile(per_query_mae_knn, 90):11.6f}")
print(f"{'P95':>10s}  ${np.percentile(per_query_mae_baseline, 95):11.6f}  ${np.percentile(per_query_mae_knn, 95):11.6f}")

Per-query MAE statistics (n=2295):
                Baseline           KNN
      Mean  $   0.039833  $   0.030579
    Median  $   0.031548  $   0.022387
       Std  $   0.027320  $   0.027717
       P90  $   0.077687  $   0.069299
       P95  $   0.101988  $   0.088458


## 7. Per-Dataset Breakdown

In [ ]:
# Per-dataset MAE breakdown for both methods
test_keys = [estimator._query_keys[i] for i in test_idx]
datasets_in_test = sorted(set(k[0] for k in test_keys))

rows = []
for ds in datasets_in_test:
    mask = np.array([k[0] == ds for k in test_keys])
    n = int(mask.sum())
    bl_mae = per_query_mae_baseline[mask].mean()
    knn_mae = per_query_mae_knn[mask].mean()
    rows.append({"Dataset": ds, "N": n,
                 "Baseline MAE ($)": f"{bl_mae:.6f}",
                 "KNN MAE ($)": f"{knn_mae:.6f}"})

ds_df = pd.DataFrame(rows)
print("=== Per-Dataset MAE Comparison (Absolute Cost) ===")
print(ds_df.to_string(index=False))

=== Per-Dataset MAE Comparison (Absolute Cost) ===
           Dataset   N Baseline MAE ($) KNN MAE ($)
       aime-hybrid  15         0.023521    0.043561
        arc-agi-v1  81         0.068918    0.058344
    arenahard-test 149         0.028406    0.030555
         gpqa-test  37         0.024826    0.029227
          hle-test 393         0.059619    0.054268
livecodebench-test 201         0.041465    0.034215
livemathbench-test  29         0.030018    0.035221
 mmlupro-test_3000 574         0.030812    0.023681
     simpleqa-test 816         0.036777    0.020034


## 8. LaTeX Tables

In [ ]:
# LaTeX Table: Per-Model MAE comparison (Mean Baseline vs KNN)

lines = []
lines.append(r"\begin{table}[t]")
lines.append(r"\centering")
lines.append(r"\caption{MAE of absolute cost estimation per model "
             r"($K{=}" + str(K) + r"$, test ratio$=$" + str(TEST_RATIO) + r"). "
             r"Costs in USD.}")
lines.append(r"\label{tab:absolute-cost-mae}")
lines.append(r"\begin{tabular}{lcc}")
lines.append(r"\toprule")
lines.append(r"Model & Mean Baseline & Embedding + KNN \\")
lines.append(r"\midrule")
for model in estimator.model_list:
    name = short_names.get(model, model)
    bl = mae_baseline[model]
    knn = mae_knn[model]
    # Bold the better (lower) value
    if bl <= knn:
        lines.append(f"{name} & \\textbf{{{bl:.4f}}} & {knn:.4f} \\\\")
    else:
        lines.append(f"{name} & {bl:.4f} & \\textbf{{{knn:.4f}}} \\\\")
lines.append(r"\midrule")
avg_bl = np.mean(list(mae_baseline.values()))
avg_knn = np.mean(list(mae_knn.values()))
if avg_bl <= avg_knn:
    lines.append(f"Average & \\textbf{{{avg_bl:.4f}}} & {avg_knn:.4f} \\\\")
else:
    lines.append(f"Average & {avg_bl:.4f} & \\textbf{{{avg_knn:.4f}}} \\\\")
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table1 = "\n".join(lines)
print(latex_table1)

print("\n" + "="*60 + "\n")

# LaTeX Table 2: Per-Dataset MAE comparison
lines2 = []
lines2.append(r"\begin{table}[t]")
lines2.append(r"\centering")
lines2.append(r"\caption{Per-dataset MAE of absolute cost estimation (USD).}")
lines2.append(r"\label{tab:absolute-cost-mae-dataset}")
lines2.append(r"\begin{tabular}{lccc}")
lines2.append(r"\toprule")
lines2.append(r"Dataset & $N$ & Mean Baseline & Embedding + KNN \\")
lines2.append(r"\midrule")
total_n = 0
for ds in datasets_in_test:
    mask = np.array([k[0] == ds for k in test_keys])
    n = int(mask.sum())
    total_n += n
    bl_val = per_query_mae_baseline[mask].mean()
    knn_val = per_query_mae_knn[mask].mean()
    if bl_val <= knn_val:
        lines2.append(f"{ds} & {n} & \\textbf{{{bl_val:.4f}}} & {knn_val:.4f} \\\\")
    else:
        lines2.append(f"{ds} & {n} & {bl_val:.4f} & \\textbf{{{knn_val:.4f}}} \\\\")
lines2.append(r"\midrule")
overall_bl = per_query_mae_baseline.mean()
overall_knn = per_query_mae_knn.mean()
if overall_bl <= overall_knn:
    lines2.append(f"Overall & {total_n} & \\textbf{{{overall_bl:.4f}}} & {overall_knn:.4f} \\\\")
else:
    lines2.append(f"Overall & {total_n} & {overall_bl:.4f} & \\textbf{{{overall_knn:.4f}}} \\\\")
lines2.append(r"\bottomrule")
lines2.append(r"\end{tabular}")
lines2.append(r"\end{table}")

latex_table2 = "\n".join(lines2)
print(latex_table2)

# Save to file
with open("../figure/absolute_cost_tables.tex", "w") as f:
    f.write("% Table 1: Per-Model MAE Comparison (Absolute Cost)\n")
    f.write(latex_table1)
    f.write("\n\n")
    f.write("% Table 2: Per-Dataset MAE Comparison (Absolute Cost)\n")
    f.write(latex_table2)
print("\nSaved to figure/absolute_cost_tables.tex")

\begin{table}[t]
\centering
\caption{MAE of absolute cost estimation per model ($K{=}5$, test ratio$=$0.2). Costs in USD.}
\label{tab:absolute-cost-mae}
\begin{tabular}{lcc}
\toprule
Model & Mean Baseline & Embedding + KNN \\
\midrule
GPT-5.2 & 0.0490 & \textbf{0.0458} \\
GPT-5 Mini & 0.0039 & \textbf{0.0027} \\
Gemini 3.1 Pro & 0.1117 & \textbf{0.0941} \\
Gemini 3 Flash & 0.0555 & \textbf{0.0344} \\
Claude Opus 4.6 & 0.0721 & \textbf{0.0520} \\
Claude Haiku 4.5 & 0.0023 & \textbf{0.0013} \\
Kimi K2.5 & 0.0233 & \textbf{0.0138} \\
MiniMax-M2.5 & 0.0008 & \textbf{0.0006} \\
\midrule
Average & 0.0398 & \textbf{0.0306} \\
\bottomrule
\end{tabular}
\end{table}


\begin{table}[t]
\centering
\caption{Per-dataset MAE of absolute cost estimation (USD).}
\label{tab:absolute-cost-mae-dataset}
\begin{tabular}{lccc}
\toprule
Dataset & $N$ & Mean Baseline & Embedding + KNN \\
\midrule
aime-hybrid & 15 & \textbf{0.0235} & 0.0436 \\
arc-agi-v1 & 81 & 0.0689 & \textbf{0.0583} \\
arenahard-test & 149 &